In [9]:
import pandas as pd
import numpy as np
import os
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy.stats import shapiro
import matplotlib.pyplot as plt
import seaborn as sns


In [10]:
path_data_clean = "../data_clean/"

file_data_bersih = os.path.join(path_data_clean, "dataset_final_2021-2023.csv")


df = pd.read_csv(file_data_bersih)
print("Data Loaded:", df.shape)
print(df.head())

cols_to_numeric = ['P1', 'UHH', 'HLS', 'RLS', 'Pengeluaran', 'TPT', 'Kepadatan']
for col in cols_to_numeric:
    df[col] = pd.to_numeric(df[col], errors='coerce')


Data Loaded: (105, 9)
                  Wilayah  Tahun    P1    UHH    HLS   RLS  Pengeluaran   TPT  \
0       Kabupaten Cilacap   2021  1.48  73.90  12.63  7.09        10534  9.97   
1      Kabupaten Banyumas   2021  2.35  73.80  13.03  7.63        11546  6.05   
2   Kabupaten Purbalingga   2021  2.10  73.21  12.00  7.25        10032  6.05   
3  Kabupaten Banjarnegara   2021  2.97  74.28  11.63  6.75         9407  5.86   
4       Kabupaten Kebumen   2021  3.24  73.55  13.35  7.55         9028  6.03   

   Kepadatan  
0        924  
1       1340  
2       1487  
3       1003  
4       1124  


Rata-rata per tahun

In [20]:
df_mean = df.groupby("Tahun")[cols_to_numeric].mean().reset_index()
display(df_mean)


,Tahun,P1,UHH,HLS,RLS,Pengeluaran,TPT,Kepadatan
0,2021,1.756286,75.012000,12.973429,7.979429,11139.200000,5.874000,2101.771429
1,2022,1.563429,75.112286,13.016286,8.141143,11533.600000,5.345429,2148.457143
2,2023,1.534286,75.233714,13.048286,8.254571,12024.428571,4.864857,2091.600000


In [22]:
train = df_mean[df_mean['Tahun'] < 2023]
test  = df_mean[df_mean['Tahun'] == 2023]

X_train = train[['UHH','HLS','RLS','Pengeluaran','TPT','Kepadatan']]
y_train = train['P1']

X_test = test[['UHH','HLS','RLS','Pengeluaran','TPT','Kepadatan']]
y_test = test['P1']

X_train_sm = sm.add_constant(X_train)
X_test_sm  = sm.add_constant(X_test)


In [23]:
model = sm.OLS(y_train, X_train_sm).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                     P1   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                    nan
Method:                 Least Squares   F-statistic:                       nan
Date:                Mon, 24 Nov 2025   Prob (F-statistic):                nan
Time:                        20:26:53   Log-Likelihood:                 62.464
No. Observations:                   2   AIC:                            -120.9
Df Residuals:                       0   BIC:                            -123.5
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const        1.187e-05        inf          0      

C:\Users\Talitha Sukma\AppData\Roaming\Python\Python312\site-packages\statsmodels\stats\stattools.py:74: ValueWarning: omni_normtest is not valid with less than 8 observations; 2 samples were given.
  warn("omni_normtest is not valid with less than 8 observations; %i "
C:\Users\Talitha Sukma\AppData\Roaming\Python\Python312\site-packages\statsmodels\regression\linear_model.py:1795: RuntimeWarning: divide by zero encountered in divide
  return 1 - (np.divide(self.nobs - self.k_constant, self.df_resid)
C:\Users\Talitha Sukma\AppData\Roaming\Python\Python312\site-packages\statsmodels\regression\linear_model.py:1795: RuntimeWarning: invalid value encountered in scalar multiply
  return 1 - (np.divide(self.nobs - self.k_constant, self.df_resid)
C:\Users\Talitha Sukma\AppData\Roaming\Python\Python312\site-packages\statsmodels\regression\linear_model.py:1717: RuntimeWarning: divide by zero encountered in scalar divide
  return np.dot(wresid, wresid) / self.df_resid


In [26]:
X_train_sm = sm.add_constant(X_train)
model = sm.OLS(y_train, X_train_sm).fit()


In [27]:
# Ambil nama kolom final pada model (termasuk constant)
cols = model.model.exog_names

# Susun test dengan kolom yang sama persis
X_test_sm = pd.DataFrame(0, index=X_test.index, columns=cols)

# Isi kolom selain constant
for c in X_test.columns:
    if c in X_test_sm.columns:
        X_test_sm[c] = X_test[c]



In [28]:
pred_test = model.predict(X_test_sm)

# MSE
mse = mean_squared_error(y_test, pred_test)

# RMSE
rmse = np.sqrt(mse)

# MAPE
mape = np.mean(np.abs((y_test - pred_test) / y_test)) * 100

# R2 (model train)
r2 = model.rsquared

print("=== METRIK TEST DATA (TAHUN 2023) ===")
print("MSE :", mse)
print("RMSE:", rmse)
print("MAPE:", mape)
print("R² (train):", r2)


=== METRIK TEST DATA (TAHUN 2023) ===
MSE : 1.5826193429932445
RMSE: 1.2580219962279056
MAPE: 81.9939848565674
R² (train): 1.0


In [29]:
df_eval = test[['Tahun', 'P1']].copy()
df_eval['Pred_P1'] = pred_test.values
df_eval['Error'] = df_eval['P1'] - df_eval['Pred_P1']
df_eval


,Tahun,P1,Pred_P1,Error
2,2023,1.534286,0.276264,1.258022
